# Niramaya Vision: DermaScan + RxRead
Run with a Colab GPU. Upload a zip containing the project checkout, or mount Drive and copy the project into `/content`. Keep `kaggle.json` private; never save it in the repository or notebook output. HAM10000 is dermatoscopic-only, and RxRead suggestions require manual verification.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
# Put the project in /content/CareCompassAi-Core_Trinity, or edit PROJECT to its Drive location.
from pathlib import Path
PROJECT = Path('/content/CareCompassAi-Core_Trinity')
assert (PROJECT / 'ml' / 'requirements-ml.txt').exists(), 'Copy/clone the project into PROJECT first.'
%cd $PROJECT
!pip -q install -r ml/requirements-ml.txt

## Prepare datasets
Upload `kaggle.json` when prompted (or configure Kaggle credentials in Colab). IAM requires its separate registration/download. If a dataset is missing, its stage will report that it cannot train.

In [ ]:
from google.colab import files
import os, shutil
uploaded = files.upload()  # select kaggle.json; keep it private
if 'kaggle.json' in uploaded:
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
!mkdir -p ml/data/ham10000 ml/data/rx_bd ml/data/lexicon ml/data/iam
# Download HAM10000 and prescription BD data only after accepting the Kaggle dataset terms.
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p ml/data/ham10000 --unzip
!kaggle datasets download -d mamun1113/doctors-handwritten-prescription-bd-dataset -p ml/data/rx_bd --unzip
# Optional India medicine lexicon; inspect the CSV and preserve its dataset terms.
!kaggle datasets download -d shudhanshusingh/az-medicine-dataset-of-india -p ml/data/lexicon --unzip
# Upload/extract IAM words.txt and words.tgz into ml/data/iam/ manually from the official IAM site.

## DermaScan training
The current trainer writes checkpoints and reports under Drive-backed paths only if the project itself is in Drive. Evaluate held-out results before exporting; synthetic/quick data is not clinical evidence.

In [ ]:
!python -m ml.skin.train --model custom
!python -m ml.skin.evaluate
!python -m ml.skin.export

## RxRead training
The IAM stage is optional. The CRNN trains on isolated words and does not read dosage instructions. Check the validation character error rate and manually inspect outputs.

In [ ]:
!python -m ml.rx.train_ctc --stage pretrain --resume
!python -m ml.rx.train_ctc --stage finetune --resume
!python -m ml.rx.export_ctc
!python -m ml.rx.evaluate_ctc

In [ ]:
import shutil
shutil.make_archive('/content/niramaya-vision-artifacts', 'zip', 'ml/artifacts')
shutil.make_archive('/content/niramaya-vision-reports', 'zip', 'ml/reports')
from google.colab import files
files.download('/content/niramaya-vision-artifacts.zip')
files.download('/content/niramaya-vision-reports.zip')